# 0 · Descarga y exploración inicial de datos

> **Proyecto:** `ClimaSafeAI`  
> **Tipo de ML:** `supervisado`  
> **Autor:** `Alejandro Cancelas Chapela`


## 1. Imports y configuración

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import missingno as msno
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import os
from dotenv import load_dotenv
from climasafeai.utils.paths import RAW_DATA_DIR, FIGURES_DIR
from climasafeai.data.make_dataset import load_data, download_era5_data, download_aemet, download_momo_data

# ── Estilo global ────────────────────────────────────S─────────────────────
plt.style.use('ggplot')
sns.set_palette('husl')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(' Imports OK')


 Imports OK


## 2. Carga de datos

Coloca el CSV en `data/raw/` y ajusta `DATA_FILE` y `TARGET_COL`.

In [6]:
load_dotenv()

download_momo_data()
download_era5_data()

    momo_data.csv ya existe en /home/cacelas/Documentos/anfaia/ClimaSafeAI/data/raw
--> Descargando ERA5 2016...


HTTPError: 403 Client Error: Forbidden for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-single-levels/execution
cost limits exceeded
Your request is too large, please reduce your selection.

In [ ]:

try:
    df = load_data(DATA_FILE)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Archivo no encontrado: {DATA_FILE}\n"
        "Coloca el CSV en data/raw/ y ajusta DATA_FILE arriba."
    )
df.head(10)


## 3. Inspección general

In [ ]:
print(f'Shape: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'Memoria: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print()

In [ ]:
df.info()

In [ ]:
# Estadísticas descriptivas completas (transpuesta para mejor lectura)
df.describe(include='all').T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

In [ ]:
# Columnas numéricas y categóricas
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()

In [ ]:

feat_num = [c for c in num_cols if c != TARGET_COL]


print(f'Numéricas ({len(num_cols)}): {num_cols}')
print(f'Categóricas ({len(cat_cols)}): {cat_cols}')

if TARGET_COL in df.columns:
    print(f'\nBalance de clases ({TARGET_COL}):')
    print(df[TARGET_COL].value_counts())
    print(df[TARGET_COL].value_counts(normalize=True).map('{:.1%}'.format))



In [ ]:
# Duplicados
n_dup = df.duplicated().sum()
print(f'Filas duplicadas: {n_dup} ({n_dup/len(df):.1%})')


## 4. Valores nulos

`missingno` usa un mapa de calor blanco/negro por filas: blanco = nulo, negro = valor presente.
Si las columnas nulas forman patrones, los nulos no son aleatorios (MNAR).

In [ ]:
null_df = pd.DataFrame({
    'Nulos': df.isnull().sum(),
    'Porcentaje': df.isnull().mean().map('{:.1%}'.format),
    'Tipo': df.dtypes
}).sort_values('Nulos', ascending=False)

null_df = null_df[null_df['Nulos'] > 0]
if null_df.empty:
    print(' Sin valores nulos')
else:
    display(null_df.style.background_gradient(cmap='Reds', subset=['Nulos']))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Matriz de nulos (missingno)
msno.matrix(df, ax=axes[0], sparkline=False, fontsize=10)
axes[0].set_title('Mapa de nulos por fila', fontsize=13)

# Barplot de % de nulos por columna
pct_null = df.isnull().mean().sort_values(ascending=False)
pct_null = pct_null[pct_null > 0]
if not pct_null.empty:
    axes[1].barh(pct_null.index, pct_null.values * 100, color='tomato', edgecolor='white')
    axes[1].axvline(20, color='orange', lw=1.5, linestyle='--', label='20% umbral')
    axes[1].axvline(50, color='red', lw=1.5, linestyle='--', label='50% umbral')
    axes[1].set_xlabel('% de nulos')
    axes[1].set_title('Porcentaje de nulos por columna')
    axes[1].legend()
    axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
else:
    axes[1].text(0.5, 0.5, 'Sin nulos ', ha='center', va='center', fontsize=14,
                 transform=axes[1].transAxes)
    axes[1].axis('off')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'missing_values.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Mapa de calor de correlación entre nulos
# Si dos columnas tienen nulos correlacionados → probablemente se originan juntos
if df.isnull().any().any():
    msno.heatmap(df, figsize=(10, 6))
    plt.title('Correlación entre nulos (Nullity Correlation)', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'nullity_heatmap.png', dpi=150)
    plt.show()


## 5. Distribuciones + KDE

Histograma + curva KDE superpuesta para detectar:
- Sesgos (skewness)
- Multimodalidad (varias 'jorobas' = posibles subpoblaciones)
- Distribuciones muy concentradas

In [ ]:
if feat_num:
    n_cols_plot = min(3, len(feat_num))
    n_rows_plot = int(np.ceil(len(feat_num) / n_cols_plot))
    fig, axes = plt.subplots(n_rows_plot, n_cols_plot,
                             figsize=(6 * n_cols_plot, 4 * n_rows_plot))
    axes = np.array(axes).ravel()

    for i, col in enumerate(feat_num):
        data = df[col].dropna()
        skew = data.skew()
        kurt = data.kurt()

        axes[i].hist(data, bins=30, density=True, alpha=0.5,
                     color='steelblue', edgecolor='white', label='Histograma')

        # KDE
        kde_x = np.linspace(data.min(), data.max(), 200)
        kde = stats.gaussian_kde(data)
        axes[i].plot(kde_x, kde(kde_x), 'r-', lw=2, label='KDE')

        # Líneas de media y mediana
        axes[i].axvline(data.mean(),   color='orange', lw=1.5, linestyle='--', label=f'Media={data.mean():.2f}')
        axes[i].axvline(data.median(), color='green',  lw=1.5, linestyle=':',  label=f'Mediana={data.median():.2f}')

        axes[i].set_title(f'{col}\nskew={skew:.2f} | kurt={kurt:.2f}', fontsize=10)
        axes[i].legend(fontsize=7)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle('Distribuciones + KDE', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'distributions_kde.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# Resumen de asimetría y curtosis
skew_kurt = pd.DataFrame({
    'Skewness': df[feat_num].skew(),
    'Kurtosis': df[feat_num].kurt()
}).sort_values('Skewness', key=abs, ascending=False)

skew_kurt.style \
    .background_gradient(cmap='RdYlGn_r', subset=['Skewness']) \
    .background_gradient(cmap='Blues',    subset=['Kurtosis']) \
    .format('{:.3f}')


## 6. Correlaciones

### 6.1 Mapa de calor 2D (clásico)

In [ ]:
corr = df[feat_num].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))  # triángulo superior

fig, ax = plt.subplots(figsize=(max(8, len(feat_num)), max(6, len(feat_num) - 1)))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Correlación de Pearson'},
    ax=ax,
)
ax.set_title('Matriz de correlación (triángulo inferior)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'correlation_matrix_2d.png', dpi=150)
plt.show()


### 6.2 Mapa de calor 3D (Plotly Surface)

In [ ]:
corr_vals = corr.values
labels    = corr.columns.tolist()
n         = len(labels)

# Superficie 3D: cada celda de la matriz de correlación se representa como altura
fig_3d = go.Figure(data=[
    go.Surface(
        z=corr_vals,
        x=labels,
        y=labels,
        colorscale='RdBu_r',
        cmin=-1, cmax=1,
        colorbar=dict(title='Correlación', thickness=15),
        hovertemplate='<b>%{x}</b> × <b>%{y}</b><br>r = %{z:.3f}<extra></extra>',
    )
])

fig_3d.update_layout(
    title=dict(text='Matriz de correlación 3D', font=dict(size=18)),
    scene=dict(
        xaxis=dict(title='', tickangle=-45, tickfont=dict(size=9)),
        yaxis=dict(title='', tickfont=dict(size=9)),
        zaxis=dict(title='Correlación', range=[-1, 1]),
        camera=dict(eye=dict(x=1.6, y=-1.6, z=1.2)),
    ),
    width=850, height=600,
    margin=dict(l=0, r=0, b=0, t=50),
)

fig_3d.show()
fig_3d.write_html(str(FIGURES_DIR / 'correlation_3d.html'))
print('Guardado: correlation_3d.html')


### 6.3 Correlaciones altas (umbral configurable)

In [ ]:
CORR_THRESHOLD = 0.75  # ← ajusta

corr_pairs = (
    corr.where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
        .stack()
        .reset_index()
)
corr_pairs.columns = ['Var1', 'Var2', 'Correlación']
corr_pairs['AbsCorr'] = corr_pairs['Correlación'].abs()
high_corr = corr_pairs[corr_pairs['AbsCorr'] >= CORR_THRESHOLD].sort_values('AbsCorr', ascending=False)

if high_corr.empty:
    print(f'Sin pares con |r| ≥ {CORR_THRESHOLD}')
else:
    print(f'Pares con |r| ≥ {CORR_THRESHOLD}:')
    display(high_corr[['Var1','Var2','Correlación']].style.background_gradient(
        cmap='RdYlGn', subset=['Correlación']
    ).format({'Correlación': '{:.3f}'}))


## 7. Detección de outliers

Se usan tres métodos complementarios:

| Método | Supuesto | Cuándo usarlo |
|---|---|---|
| **IQR** | No paramétrico | Siempre, especialmente con sesgos |
| **Z-score** | Distribución normal | Variables aproximadamente normales |
| **Isolation Forest** | Sin supuestos | Outliers multivariantes complejos |

### 7.1 Boxplots (visión rápida)

In [ ]:
if feat_num:
    n_cols_plot = min(4, len(feat_num))
    n_rows_plot = int(np.ceil(len(feat_num) / n_cols_plot))
    fig, axes = plt.subplots(n_rows_plot, n_cols_plot,
                             figsize=(5 * n_cols_plot, 4 * n_rows_plot))
    axes = np.array(axes).ravel()

    for i, col in enumerate(feat_num):
        data = df[col].dropna()
        q1, q3 = data.quantile(0.25), data.quantile(0.75)
        iqr  = q3 - q1
        low  = q1 - 1.5 * iqr
        high = q3 + 1.5 * iqr
        n_out = ((data < low) | (data > high)).sum()

        bp = axes[i].boxplot(data, vert=True, patch_artist=True,
                             flierprops=dict(marker='o', markerfacecolor='red',
                                            markersize=4, alpha=0.6))
        bp['boxes'][0].set_facecolor('steelblue')
        bp['boxes'][0].set_alpha(0.6)
        axes[i].set_title(f'{col}\n({n_out} outliers IQR)', fontsize=9)
        axes[i].set_xticklabels([])

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle('Boxplots — outliers marcados en rojo', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'boxplots.png', dpi=150)
    plt.show()


### 7.2 IQR — resumen cuantitativo

In [ ]:
iqr_results = []
for col in feat_num:
    data = df[col].dropna()
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr  = q3 - q1
    low  = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    mask_out = (data < low) | (data > high)
    iqr_results.append({
        'Variable':    col,
        'Q1':          round(q1, 3),
        'Q3':          round(q3, 3),
        'IQR':         round(iqr, 3),
        'Límite_inf':  round(low, 3),
        'Límite_sup':  round(high, 3),
        'N_outliers':  int(mask_out.sum()),
        'Pct_outliers': f'{mask_out.mean():.1%}',
    })

df_iqr = pd.DataFrame(iqr_results).sort_values('N_outliers', ascending=False)
df_iqr.style.background_gradient(cmap='Oranges', subset=['N_outliers'])


### 7.3 Z-score (umbral = 3)

In [ ]:
Z_THRESHOLD = 3  # ← outlier si |z| > 3

z_results = []
for col in feat_num:
    data = df[col].dropna()
    z    = np.abs(stats.zscore(data))
    n_out = (z > Z_THRESHOLD).sum()
    z_results.append({'Variable': col, 'N_outliers_z': int(n_out),
                      'Pct': f'{n_out/len(data):.1%}',
                      'Max_|z|': round(z.max(), 2)})

df_z = pd.DataFrame(z_results).sort_values('N_outliers_z', ascending=False)
df_z.style.background_gradient(cmap='Reds', subset=['N_outliers_z'])


### 7.4 Isolation Forest — outliers multivariantes 

Isolation Forest detecta outliers que son anómalos en el **espacio multidimensional** completo,
no variable a variable. Es especialmente útil para detectar combinaciones raras de valores
que individualmente parecen normales.

In [ ]:
CONTAMINATION = 0.05  # ← % esperado de outliers en el dataset

X_num = df[feat_num].dropna()
scaler_iso = StandardScaler()
X_scaled   = scaler_iso.fit_transform(X_num)

iso = IsolationForest(
    contamination=CONTAMINATION,
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)
iso.fit(X_scaled)

scores    = iso.decision_function(X_scaled)   # más negativo = más outlier
labels_if = iso.predict(X_scaled)             # -1 = outlier, 1 = normal

n_outliers = (labels_if == -1).sum()
print(f'Isolation Forest → {n_outliers} outliers detectados ({n_outliers/len(X_num):.1%})')

df_iso = X_num.copy()
df_iso['iso_score']  = scores
df_iso['iso_outlier'] = labels_if == -1


In [ ]:
# Visualización del anomaly score
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma del score
axes[0].hist(scores[labels_if ==  1], bins=40, alpha=0.7, color='steelblue', label='Normal')
axes[0].hist(scores[labels_if == -1], bins=40, alpha=0.7, color='red',       label='Outlier')
axes[0].axvline(0, color='black', lw=1.5, linestyle='--', label='Umbral=0')
axes[0].set_xlabel('Anomaly Score')
axes[0].set_ylabel('Muestras')
axes[0].set_title('Distribución del Anomaly Score')
axes[0].legend()

# PCA 2D para visualizar separación
from sklearn.decomposition import PCA
pca_iso = PCA(n_components=2)
X_pca   = pca_iso.fit_transform(X_scaled)

axes[1].scatter(X_pca[labels_if ==  1, 0], X_pca[labels_if ==  1, 1],
                c='steelblue', s=10, alpha=0.5, label='Normal')
axes[1].scatter(X_pca[labels_if == -1, 0], X_pca[labels_if == -1, 1],
                c='red', s=30, alpha=0.8, marker='x', label='Outlier', linewidths=1.5)
axes[1].set_xlabel(f'PC1 ({pca_iso.explained_variance_ratio_[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({pca_iso.explained_variance_ratio_[1]:.1%})')
axes[1].set_title('Outliers Isolation Forest — PCA 2D')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'isolation_forest.png', dpi=150)
plt.show()


In [ ]:
# Top 10 outliers más extremos
top_outliers = df_iso[df_iso['iso_outlier']].sort_values('iso_score').head(10)
print('Top 10 outliers más extremos:')
display(top_outliers.drop(columns=['iso_score','iso_outlier']).style.highlight_max(axis=0, color='#ffcccc'))


### 7.5 Comparativa IQR vs Z-score vs Isolation Forest

In [ ]:
# Comparativa del número de outliers detectados por cada método
compare = df_iqr[['Variable','N_outliers']].rename(columns={'N_outliers': 'IQR'})
compare = compare.merge(
    df_z[['Variable','N_outliers_z']].rename(columns={'N_outliers_z': 'Z-score'}),
    on='Variable'
).set_index('Variable')

fig, ax = plt.subplots(figsize=(max(8, len(feat_num)), 5))
compare.plot(kind='bar', ax=ax, edgecolor='white', width=0.7)
ax.set_title('Outliers detectados por método (IQR vs Z-score)', fontsize=13)
ax.set_ylabel('Nº de outliers')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'outliers_comparison.png', dpi=150)
plt.show()


## 8. Violin plots

Combinan la información del boxplot con la distribución real (KDE lateral).
Son especialmente útiles cuando hay multimodalidad que el boxplot no muestra.

In [ ]:
if feat_num:
    n_cols_plot = min(3, len(feat_num))
    n_rows_plot = int(np.ceil(len(feat_num) / n_cols_plot))
    fig, axes = plt.subplots(n_rows_plot, n_cols_plot,
                             figsize=(6 * n_cols_plot, 5 * n_rows_plot))
    axes = np.array(axes).ravel()

    
    hue_col = TARGET_COL if TARGET_COL and TARGET_COL in df.columns else None
    

    for i, col in enumerate(feat_num):
        if hue_col:
            sns.violinplot(
                data=df, x=hue_col, y=col,
                ax=axes[i], inner='box', palette='husl',
                cut=0, density_norm='width',
            )
            axes[i].set_title(f'{col} por {hue_col}', fontsize=10)
        else:
            sns.violinplot(
                data=df, y=col, ax=axes[i],
                inner='box', color='steelblue', cut=0,
            )
            axes[i].set_title(col, fontsize=10)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle('Violin plots', fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'violin_plots.png', dpi=150)
    plt.show()


## 9. Pairplot interactivo (Plotly)

Scatter matrix de todas las variables numéricas. Permite detectar:
- Relaciones lineales y no lineales
- Grupos o clusters visuales
- Outliers en el espacio 2D de cada par de variables

In [ ]:
MAX_PAIR_FEATURES = 6  # limitar para que no sea lentísimo con muchas variables
pair_cols = feat_num[:MAX_PAIR_FEATURES]


color_col = TARGET_COL if TARGET_COL and TARGET_COL in df.columns else None


fig_pair = px.scatter_matrix(
    df[pair_cols + ([color_col] if color_col else [])].dropna(),
    dimensions=pair_cols,
    color=color_col,
    title='Scatter Matrix (pairplot interactivo)',
    opacity=0.5,
    height=800,
)
fig_pair.update_traces(diagonal_visible=False, marker=dict(size=3))
fig_pair.show()
fig_pair.write_html(str(FIGURES_DIR / 'pairplot_interactive.html'))
print('Guardado: pairplot_interactive.html')


In [ ]:
# Pairplot estático seaborn (para guardar PNG)

hue_pair = TARGET_COL if TARGET_COL and TARGET_COL in df.columns else None


g = sns.pairplot(
    df[pair_cols + ([hue_pair] if hue_pair else [])].dropna(),
    hue=hue_pair,
    diag_kind='kde',
    plot_kws={'alpha': 0.4, 's': 15},
    diag_kws={'fill': True},
)
g.figure.suptitle('Pairplot — primeras variables numéricas', y=1.02, fontsize=13)
g.figure.savefig(FIGURES_DIR / 'pairplot.png', dpi=120, bbox_inches='tight')
plt.show()


## 10. Variables categóricas

In [ ]:

plot_cat = [c for c in cat_cols if c != TARGET_COL]


if plot_cat:
    n_cols_plot = min(3, len(plot_cat))
    n_rows_plot = int(np.ceil(len(plot_cat) / n_cols_plot))
    fig, axes = plt.subplots(n_rows_plot, n_cols_plot,
                             figsize=(6 * n_cols_plot, 4 * n_rows_plot))
    axes = np.array(axes).ravel()

    for i, col in enumerate(plot_cat):
        vc = df[col].value_counts().head(15)  # top 15 valores
        vc.plot(kind='barh', ax=axes[i], color='steelblue', edgecolor='white')
        axes[i].set_title(f'{col} (top 15)', fontsize=10)
        axes[i].set_xlabel('Frecuencia')

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle('Distribución de variables categóricas', fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'categorical.png', dpi=150)
    plt.show()
else:
    print('Sin variables categóricas')


In [ ]:

# Categóricas vs target: stacked bar normalizado
if plot_cat and TARGET_COL and TARGET_COL in df.columns:
    n_cols_plot = min(2, len(plot_cat))
    n_rows_plot = int(np.ceil(len(plot_cat) / n_cols_plot))
    fig, axes = plt.subplots(n_rows_plot, n_cols_plot,
                             figsize=(8 * n_cols_plot, 5 * n_rows_plot))
    axes = np.array(axes).ravel()

    for i, col in enumerate(plot_cat):
        ct = pd.crosstab(df[col], df[TARGET_COL], normalize='index') * 100
        ct.plot(kind='bar', stacked=True, ax=axes[i], edgecolor='white', width=0.8)
        axes[i].set_title(f'{col} vs {TARGET_COL}', fontsize=10)
        axes[i].set_ylabel('%')
        axes[i].tick_params(axis='x', rotation=45)
        axes[i].yaxis.set_major_formatter(mtick.PercentFormatter())

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f'Categóricas vs {TARGET_COL} (stacked %)', fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'categorical_vs_target.png', dpi=150)
    plt.show()



## 11. Resumen automático de calidad del dataset

In [ ]:
total_cells = df.shape[0] * df.shape[1]
null_cells  = df.isnull().sum().sum()
dup_rows    = df.duplicated().sum()

# Outliers IQR totales
total_iqr_out = sum(
    ((df[col] < df[col].quantile(0.25) - 1.5*(df[col].quantile(0.75)-df[col].quantile(0.25))) |
     (df[col] > df[col].quantile(0.75) + 1.5*(df[col].quantile(0.75)-df[col].quantile(0.25)))).sum()
    for col in feat_num
)

high_corr_pairs = len(high_corr)
skewed_cols = (df[feat_num].skew().abs() > 1).sum()

print('═' * 50)
print('   RESUMEN DE CALIDAD DEL DATASET')
print('═' * 50)
print(f'  Filas × columnas   : {df.shape[0]:,} × {df.shape[1]}')
print(f'  Memoria            : {df.memory_usage(deep=True).sum()/1024**2:.2f} MB')
print(f'  Celdas nulas       : {null_cells:,} ({null_cells/total_cells:.1%})')
print(f'  Filas duplicadas   : {dup_rows:,} ({dup_rows/len(df):.1%})')
print(f'  Outliers IQR total : {total_iqr_out:,}')
print(f'  Outliers Iso.Forest: {n_outliers:,} ({n_outliers/len(df):.1%})')
print(f'  Pares alta corr.   : {high_corr_pairs} (|r|≥{CORR_THRESHOLD})')
print(f'  Variables sesgadas : {skewed_cols} (|skew|>1)')

if TARGET_COL and TARGET_COL in df.columns:
    vc = df[TARGET_COL].value_counts(normalize=True)
    imbalance = vc.max() / vc.min() if len(vc) > 1 else 1
    print(f'  Desbalanceo target : {imbalance:.1f}x ({vc.idxmax()} domina)')

print('═' * 50)
